# Guide pour l'Appel à une API Hugging Face pour la Segmentation d'Images

Bienvenue ! Ce notebook a pour but de vous guider pas à pas dans l'utilisation de l'API d'inférence de Hugging Face pour effectuer de la segmentation d'images. La segmentation d'images consiste à attribuer une étiquette (comme "cheveux", "vêtement", "arrière-plan") à chaque pixel d'une image.

Nous allons :
1. Comprendre ce qu'est une API et comment s'y connecter.
2. Envoyer une image à un modèle de segmentation hébergé sur Hugging Face.
3. Récupérer et interpréter les résultats.
4. Visualiser les masques de segmentation.
5. Étendre cela pour traiter plusieurs images.

## 1. Configuration Initiale et Importations

Commençons par importer les bibliothèques Python nécessaires. Nous aurons besoin de :
- `os` pour interagir avec le système de fichiers (lister les images).
- `requests` pour effectuer des requêtes HTTP vers l'API.
- `PIL (Pillow)` pour manipuler les images.
- `matplotlib.pyplot` pour afficher les images et les masques.
- `numpy` pour la manipulation des tableaux (les images sont des tableaux de pixels).
- `tqdm.notebook` pour afficher une barre de progression (utile pour plusieurs images).
- `base64` et `io` pour décoder les masques renvoyés par l'API.

In [13]:
import os
import requests
from PIL import Image # la référence en Python pour manipuler les images (Ouvre l'image stockée sur disque dur)
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import base64
import io # sert à manipuler des données en mémoire. io.StringIO : Permet de lire et écrire en str en mémoire vive, io.BufferedReader / io.BufferedWriter "tampon" (buffer) accélèrent la lecture et l'écriture de gros fichiers 
            # io.BytesIO : Fonctionne exactement comme StringIO, mais pour les données binaires (bytes)
from dotenv import load_dotenv # Pour récupérer le token dans le .env
import time # sert à gérer le temps, mesurer des durées et manipuler des dates. time.sleep(secondes) arrête l'exécution, time.time() donne le nombre de secondes écoulées, time.localtime() ou time.strftime() permettent de récupérer l'heure

### Variables de Configuration

Nous devons définir quelques variables :
- `image_dir`: Le chemin vers le dossier contenant vos images. **Assurez-vous de modifier ce chemin si nécessaire.**
- `max_images`: Le nombre maximum d'images à traiter (pour ne pas surcharger l'API ou attendre trop longtemps).
- `api_token`: Votre jeton d'API Hugging Face. **IMPORTANT : Gardez ce jeton secret !**

**Comment obtenir un token API Hugging Face ?**
1. Créez un compte sur [huggingface.co](https://huggingface.co/).
2. Allez dans votre profil -> Settings -> Access Tokens.
3. Créez un nouveau token (par exemple, avec le rôle "read").
4. Copiez ce token ici.

In [4]:
# Charger le token depuis le fichier .env
load_dotenv()
# Puis on vérifie que le token est bien chargé 
HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("Le token n'a pas été trouvé. fichier .env à vérifier")
else:
    print("Token Hugging Face chargé avec succès.")

Token Hugging Face chargé avec succès.


In [5]:
# TODO: Modifiez ces valeurs selon votre configuration
image_dir = r"D:\Formation ai Engineer OpenClassRooms\Projet 2 Requêtez des services IA\top_influenceurs_2024\IMG"  # Exemple : si vous êtes sur Colab et avez uploadé un dossier, r pour raw
max_images = 3  # Commençons avec peu d'images

# IMPORTANT: Remplacez "VOTRE_TOKEN_HUGGING_FACE_ICI" par votre véritable token API.
# Ne partagez jamais votre token publiquement.
api_token = HF_TOKEN

# Créons le dossier d'images s'il n'existe pas (pour l'exemple)
if not os.path.exists(image_dir):
    os.makedirs(image_dir)
    print(f"Dossier '{image_dir}' créé. Veuillez y ajouter des images .jpg ou .png.")
else:
    print(f"Dossier '{image_dir}' existant.")

if api_token == HF_TOKEN:
    print("\nATTENTION : Vous devez remplacer 'VOTRE_TOKEN_HUGGING_FACE_ICI' par votre token API personnel.")

Dossier 'D:\Formation ai Engineer OpenClassRooms\Projet 2 Requêtez des services IA\top_influenceurs_2024\IMG' existant.

ATTENTION : Vous devez remplacer 'VOTRE_TOKEN_HUGGING_FACE_ICI' par votre token API personnel.


## 2. Comprendre l'API d'Inférence Hugging Face

L'API d'inférence permet d'utiliser des modèles hébergés sur Hugging Face sans avoir à les télécharger ou à gérer l'infrastructure.

- **Modèle utilisé** : Nous allons utiliser le modèle `sayeed99/segformer_b3_clothes`, spécialisé dans la segmentation de vêtements et de parties du corps.
- **URL de l'API** : L'URL pour un modèle est généralement `https://api-inference.huggingface.co/models/NOM_DU_MODELE`.
- **Headers (En-têtes)** : Pour s'authentifier et spécifier le type de contenu, nous envoyons des en-têtes avec notre requête.
    - `Authorization`: Contient notre token API (précédé de `Bearer `).
    - `Content-Type`: Indique que nous envoyons une image au format JPEG (ou PNG selon le cas).

In [7]:
API_URL = "https://router.huggingface.co/hf-inference/models/sayeed99/segformer_b3_clothes" # Remplacez ... par le bon endpoint.
headers = {
    "Authorization": f"Bearer {api_token}"
    # Le "Content-Type" sera ajouté dynamiquement lors de l'envoi de l'image
}

# Lister les chemins des images à traiter
# Assurez-vous d'avoir des images dans le dossier 'image_dir'!
image_paths = [
    os.path.join(image_dir, f) 
    for f in os.listdir(image_dir) 
    if f.lower().endswith(".png")
              ]


if not image_paths:
    print(f"Aucune image trouvée dans '{image_dir}'. Veuillez y ajouter des images.")
else:
    print(f"{len(image_paths)} image(s) à traiter : {image_paths}")

50 image(s) à traiter : ['D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_0.png', 'D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_1.png', 'D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_10.png', 'D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_11.png', 'D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_12.png', 'D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_13.png', 'D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_14.png', 'D:\\Formation ai Engineer OpenClassRooms\\Projet 2 Requêtez des services IA\\top_influenceurs_2024\\IMG\\image_15.png', 'D:\\Form

## 3. Fonctions Utilitaires pour le Traitement des Masques

Le modèle que nous utilisons (`sayeed99/segformer_b3_clothes`) renvoie des masques pour différentes classes (cheveux, chapeau, etc.). Ces masques sont encodés en base64. Les fonctions ci-dessous sont fournies pour vous aider à :
1.  `CLASS_MAPPING`: Un dictionnaire qui associe les noms de classes (ex: "Hat") à des identifiants numériques.
2.  `get_image_dimensions`: Récupérer les dimensions d'une image.
3.  `decode_base64_mask`: Décoder un masque de base64 en une image (tableau NumPy) et le redimensionner.
4.  `create_masks`: Combiner les masques de toutes les classes détectées en un seul masque de segmentation final, où chaque pixel a la valeur de l'ID de sa classe.

**Cette partie est donnée car elle est spécifique au format de sortie de ce modèle et un peu complexe pour une première approche.** Lisez-la pour comprendre son rôle, mais ne vous attardez pas sur les détails d'implémentation pour l'instant.

In [8]:
CLASS_MAPPING = {
    "Background": 0,
    "Hat": 1,
    "Hair": 2,
    "Sunglasses": 3,
    "Upper-clothes": 4,
    "Skirt": 5,
    "Pants": 6,
    "Dress": 7,
    "Belt": 8,
    "Left-shoe": 9,
    "Right-shoe": 10,
    "Face": 11,
    "Left-leg": 12,
    "Right-leg": 13,
    "Left-arm": 14,
    "Right-arm": 15,
    "Bag": 16,
    "Scarf": 17
}

def get_image_dimensions(img_path):
    """
    Get the dimensions of an image.

    Args:
        img_path (str): Path to the image.

    Returns:
        tuple: (width, height) of the image.
    """
    original_image = Image.open(img_path)
    return original_image.size

def decode_base64_mask(base64_string, width, height):
    """
    Decode a base64-encoded mask into a NumPy array.

    Args:
        base64_string (str): Base64-encoded mask.
        width (int): Target width.
        height (int): Target height.

    Returns:
        np.ndarray: Single-channel mask array.
    """
    mask_data = base64.b64decode(base64_string)
    mask_image = Image.open(io.BytesIO(mask_data))
    mask_array = np.array(mask_image)
    if len(mask_array.shape) == 3:
        mask_array = mask_array[:, :, 0]  # Take first channel if RGB
    mask_image = Image.fromarray(mask_array).resize((width, height), Image.NEAREST)
    return np.array(mask_image)

def create_masks(results, width, height):
    """
    Combine multiple class masks into a single segmentation mask.

    Args:
        results (list): List of dictionaries with 'label' and 'mask' keys.
        width (int): Target width.
        height (int): Target height.

    Returns:
        np.ndarray: Combined segmentation mask with class indices.
    """
    combined_mask = np.zeros((height, width), dtype=np.uint8)  # Initialize with Background (0)

    # Process non-Background masks first
    for result in results:
        label = result['label']
        class_id = CLASS_MAPPING.get(label, 0)
        if class_id == 0:  # Skip Background
            continue
        mask_array = decode_base64_mask(result['mask'], width, height)
        combined_mask[mask_array > 0] = class_id

    # Process Background last to ensure it doesn't overwrite other classes unnecessarily
    # (Though the model usually provides non-overlapping masks for distinct classes other than background)
    for result in results:
        if result['label'] == 'Background':
            mask_array = decode_base64_mask(result['mask'], width, height)
            # Apply background only where no other class has been assigned yet
            # This logic might need adjustment based on how the model defines 'Background'
            # For this model, it seems safer to just let non-background overwrite it first.
            # A simple application like this should be fine: if Background mask says pixel is BG, set it to 0.
            # However, a more robust way might be to only set to background if combined_mask is still 0 (initial value)
            combined_mask[mask_array > 0] = 0 # Class ID for Background is 0

    return combined_mask

## 4. Segmentation d'une Seule Image

Avant de traiter toutes les images, concentrons-nous sur une seule pour bien comprendre le processus.

Étapes :
1.  Choisir une image.
2.  Ouvrir l'image en mode binaire (`"rb"`) et lire son contenu (`data`).
3.  Déterminer le `Content-Type` (par exemple, `"image/jpeg"` ou `"image/png"`).
4.  Envoyer la requête POST à l'API avec `requests.post()` en passant l'URL, les headers et les données.
5.  Vérifier le code de statut de la réponse. Une erreur sera levée si le code n'est pas 2xx (succès) grâce à `response.raise_for_status()`.
6.  Convertir la réponse JSON en un dictionnaire Python avec `response.json()`.
7.  Utiliser nos fonctions `get_image_dimensions` et `create_masks` pour obtenir le masque final.
8.  Afficher l'image originale et le masque segmenté.

In [14]:
if image_paths:
    single_image_path = image_paths[0] # Prenons la première image de notre liste
    print(f"Traitement de l'image : {single_image_path}")

    try:
        # Lire l'image en binaire
        # Et mettez le contenu de l'image dans la variable image_data
        # Redimension du PNG et passage en JPEG compressé (évite les timeouts et dépassements de taille)
        with Image.open(single_image_path) as img:
            img = img.convert("RGB")
            # Conserve le ratio d'aspect tout en limitant la résolution à 512px max
            img.thumbnail((512, 512))
            buffer = io.BytesIO() # fichier temporaire en mémoire vive
            img.save(buffer, format="JPEG", quality=85) # les images sont mis dans le fichier temporaire en JPEG
            image_data = buffer.getvalue() # extrait les octets binaires bruts de ce fichier temporaire 

        # Maintenant, utilisé l'API huggingface : Appel à l'api avec l'authentification par le header qui contient le token
        headers_requete = {
            **headers,
            "Content-Type": "image/jpeg"
        }
        print("Envoi de la requête à l'API SegFormer...")
        response = requests.post(API_URL, headers=headers_requete, data=image_data, timeout=45)
        # ainsi que les fonctions données plus haut pour ségmenter vos images.
        
        # Gestion du modèle en cours de chargement sur l'infra serverless si pas de retour serveur
        if response.status_code == 503:
            temps_estime = response.json().get("estimated_time", 20.0)
            print(f"Modèle en cours d'initialisation (503). Attente de {temps_estime:.1f} secondes...")
            time.sleep(temps_estime)
            # Deuxième tentative une fois le modèle prêt
            response = requests.post(API_URL, headers=headers_requete, data=image_data, timeout=45)

        # Déclenche une exception si le statut HTTP indique une erreur (401, 404, 500, etc.)
        response.raise_for_status()
        results = response.json()
        print("✓ Réponse reçue de l'API avec succès.")

        # Récupération des dimensions réelles et génération du masque combiné
        width, height = get_image_dimensions(single_image_path)
        combined_mask = create_masks(results, width, height)
        
        classes_detectees = np.unique(combined_mask)
        print(f"✓ Masque généré avec succès.")
        print(f"  - Dimensions : {combined_mask.shape}")
        print(f"  - Classes trouvées (IDs) : {classes_detectees}")

    except requests.exceptions.Timeout:
        print("Erreur : La requête a expiré (Timeout dépassé).")
    except requests.exceptions.HTTPError as http_err:
        print(f"Erreur HTTP retournée par l'API : {http_err} - Détail : {response.text}")
    except Exception as e:
        print(f"Une erreur est survenue lors du traitement : {e}")
else:
    print("Aucune image à traiter. Vérifiez la configuration de 'image_dir' et 'max_images'.")

Traitement de l'image : D:\Formation ai Engineer OpenClassRooms\Projet 2 Requêtez des services IA\top_influenceurs_2024\IMG\image_0.png
Envoi de la requête à l'API SegFormer...
Erreur HTTP retournée par l'API : 403 Client Error: Forbidden for url: https://router.huggingface.co/hf-inference/models/sayeed99/segformer_b3_clothes - Détail : {"error":"This authentication method does not have sufficient permissions to call Inference Providers on behalf of user Alexis-h2"}


## 5. Segmentation de Plusieurs Images (Batch)

Maintenant que nous savons comment traiter une image, nous pouvons créer une fonction pour en traiter plusieurs.
Cette fonction va boucler sur la liste `image_paths` et appliquer la logique de segmentation à chaque image.
Nous utiliserons `tqdm` pour avoir une barre de progression.

In [10]:
def segment_images_batch(list_of_image_paths):
    """
    Segmente une liste d'images en utilisant l'API Hugging Face.

    Args:
        list_of_image_paths (list): Liste des chemins vers les images.

    Returns:
        list: Liste des masques de segmentation (tableaux NumPy).
              Contient None si une image n'a pas pu être traitée.
    """
    batch_segmentations = []

    # N'oubliez pas de mettre une pause entre chaque appel API !


    return batch_segmentations

# Appeler la fonction pour segmenter les images listées dans image_paths
if image_paths:
    print(f"\nTraitement de {len(image_paths)} image(s) en batch...")
    batch_seg_results = segment_images_batch(image_paths)
    print("Traitement en batch terminé.")
else:
    batch_seg_results = []
    print("Aucune image à traiter en batch.")


Traitement de 50 image(s) en batch...
Traitement en batch terminé.


## 6. Affichage des Résultats en Batch

Nous allons maintenant créer une fonction pour afficher les images originales et leurs segmentations correspondantes côte à côte, dans une grille.

In [11]:
def display_segmented_images_batch(original_image_paths, segmentation_masks):
    """
    Affiche les images originales et leurs masques segmentés.

    Args:
        original_image_paths (list): Liste des chemins des images originales.
        segmentation_masks (list): Liste des masques segmentés (NumPy arrays).
    """
    # Matplotlib, ça vous parle ?
    # Alors... au travail ! 😉

# Afficher les résultats du batch
if batch_seg_results:
    display_segmented_images_batch(image_paths, batch_seg_results)
else:
    print("Aucun résultat de segmentation à afficher.")

Aucun résultat de segmentation à afficher.


## Conclusion et Prochaines Étapes

Félicitations ! Vous avez appris à :
- Configurer les appels à l'API d'inférence Hugging Face.
- Envoyer des images pour la segmentation.
- Interpréter les résultats (avec l'aide des fonctions fournies).
- Visualiser les segmentations.

Pistes d'amélioration ou d'exploration :
- **Gestion d'erreurs plus fine** : Implémenter des tentatives multiples (retry) en cas d'échec de l'API (par exemple, si le modèle est en cours de chargement).
- **Appels asynchrones** : Pour un grand nombre d'images, des appels asynchrones (avec `asyncio` et `aiohttp`) seraient beaucoup plus rapides.
- **Autres modèles** : Explorer d'autres modèles de segmentation ou d'autres tâches sur Hugging Face Hub.

N'hésitez pas à modifier le code, à tester avec vos propres images et à explorer davantage !

**_Note_** : Si vous aimez ce modèle, n'hésitez pas à le [télécharger](https://huggingface.co/sayeed99/segformer_b3_clothes) et jouer avec directement sur votre machine !